# Production Method Comparison (Economic Analysis)

Live economic comparison of vanilla and modded production methods, grouped by slots.

In [31]:
import pandas as pd
import numpy as np

from core.parser.path_resolver import PathResolver
from core.data.building_data import BuildingData
from core.data.goods_data import GoodsData
from core.data.defines_data import DefinesData
from core.data.pop_data import PopData
from analysis.building_levels.building_analysis import load_config

config = load_config()
path_resolver = PathResolver(config['game_path'], config['mod_path'])
goods_data = GoodsData(path_resolver)
building_data = BuildingData(path_resolver)
defines_data = DefinesData(path_resolver)
pop_data = PopData(path_resolver)

goods_data.load_all()
building_data.load_all()
defines_data.load_all()
pop_data.load_all()

pd.options.display.max_columns = None
pd.options.display.precision = 2
pd.options.display.float_format = '{:.2f}'.format

In [32]:
# Resolved food good (good with food > 0) and its price; fallback to defines
food_good_vanilla = goods_data.get_food_good(False)
food_good_modded = goods_data.get_food_good(True)
food_price_vanilla = goods_data.get_food_good_price(False)
food_price_modded = goods_data.get_food_good_price(True)
if food_price_vanilla is None:
    food_price_vanilla = defines_data.get_vanilla_define("NMarket", "FOOD_PRICE")
if food_price_modded is None:
    food_price_modded = defines_data.get_define("NMarket", "FOOD_PRICE")
print(f"Vanilla food good: {food_good_vanilla}, price: {food_price_vanilla}")
print(f"Modded food good:  {food_good_modded}, price: {food_price_modded}")

Vanilla food good: rice, price: 1.0
Modded food good:  None, price: 0.05


In [33]:
pop_comparison = pd.merge(
    pop_data.vanilla_df[['pop_food_consumption']].rename(columns={'pop_food_consumption': 'Vanilla_Food'}),
    pop_data.modded_df[['pop_food_consumption']].rename(columns={'pop_food_consumption': 'Modded_Food'}),
    left_index=True,
    right_index=True
)
pop_comparison['Change'] = (pop_comparison['Modded_Food'] / pop_comparison['Vanilla_Food']) - 1
pop_comparison.style.format({'Change': '{:+.1%}'}).background_gradient(subset=['Change'], cmap='RdYlGn_r')

,Vanilla_Food,Modded_Food,Change
name,,,
nobles,20.000000,20.000000,+0.0%
clergy,5.000000,5.000000,+0.0%
burghers,4.000000,5.000000,+25.0%
laborers,1.000000,1.500000,+50.0%
soldiers,5.000000,2.500000,-50.0%
peasants,1.000000,1.000000,+0.0%
tribesmen,0.000000,0.000000,+nan%
slaves,1.000000,1.000000,+0.0%


In [56]:
# goods_data.vanilla_df.columns
# print(goods_data.vanilla_df.shape)
goods_data.vanilla_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).head(14)
# goods_data.vanilla_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).head(40)


for element in goods_data.vanilla_df.reset_index().name.tolist():
    print(element)

horses
clay
sand
coal
iron
copper
goods_gold
silver
stone
tin
lead
silk
dyes
incense
tea
cocoa
coffee
fiber_crops
ivory
lumber
salt
medicaments
gems
pearls
amber
saltpeter
alum
wine
elephants
marble
mercury
saffron
pepper
cloves
chili
cotton
sugar
tobacco
tar
porcelain
naval_supplies
firearms
cannons
weaponry
glass
steel
cloth
fine_cloth
liquor
beer
paper
books
jewelry
leather
tools
masonry
lacquerware
pottery
furniture
wool
wild_game
fur
fish
wheat
maize
rice
millet
legumes
potato
livestock
olives
fruit
beeswax
slaves_goods


In [35]:
# goods_data.modded_df.columns
# print(goods_data.modded_df.shape)
# print(goods_data.modded_df.columns)
# goods_data.modded_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).tail(40)
# goods_data.modded_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="transport_cost", ascending=True).head(40)

In [36]:
def get_slot_df(building_name, slot_index, is_modded=True):
    comp = building_data.compare_production_methods(building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data)
    slots = comp['modded_slots'] if is_modded else comp['vanilla_slots']
    
    if slot_index >= len(slots):
        return pd.DataFrame()
    
    # Get building definition for employment info
    b_def = building_data.get_building(building_name) if is_modded else building_data.get_vanilla_building(building_name)
    b_def = b_def or {}
    
    slot = slots[slot_index]
    rows = []
    # System columns that shouldn't be prefixed with In_
    system_cols = [
        'profit', 'profit_margin', 'input_cost', 'output_value', 
        'worker_food_cost', 'is_modifier_output', 'output_price', 'modifier_food_output'
    ]
    
    for pm_name, pm in slot.items():
        row = {
            "Version": "Modded" if is_modded else "Vanilla",
            "PM": pm_name,
            "Pop_Type": b_def.get('pop_type', 'unknown'),
            "Employment (1k)": b_def.get('employment_size_val', 0),
            "EPE": pm.get('epe', 0),
            "Profit": pm.get('profit', 0),
            "Margin": pm.get('profit_margin', 0),
            "Input_Cost": pm.get('input_cost', 0),
            "Output_Val": pm.get('output_value', 0),
            "Worker_Food_Cost": pm.get('worker_food_cost', 0)
        }
        for key, val in pm.items():
            if key in ['produced', 'output', 'category']:
                continue
            
            if key in system_cols:
                row[key] = val
            elif not key.startswith(('profit', 'input_cost', 'output_value', 'worker_food_cost', 'output_price', 'is_modifier_output', 'epe')):
                # Prefix goods with In_
                row[f"In_{key}"] = val
                
        # Handle Output columns
        prod = pm.get('produced')
        if prod and not pd.isna(prod):
            row[f"Out_{prod}"] = pm.get('output', 0)
        modifier_food = pm.get('modifier_food_output', 0)
        if modifier_food and modifier_food != 0:
            row["Out_food"] = modifier_food
            
        rows.append(row)
    return pd.DataFrame(rows)

def analyze_building(building_name):
    comp = building_data.compare_production_methods(building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data)
    num_slots = max(len(comp['vanilla_slots']), len(comp['modded_slots']))
    
    slot_dfs = []
    for i in range(num_slots):
        v_df = get_slot_df(building_name, i, is_modded=False)
        m_df = get_slot_df(building_name, i, is_modded=True)
        combined = pd.concat([v_df, m_df], ignore_index=True)
        if combined.empty: continue
        
        # Cleanup zero columns
        for col in combined.columns:
            if col.startswith(('In_', 'Out_')):
                if (combined[col].fillna(0).abs() < 1e-6).all():
                    combined = combined.drop(columns=[col])
        
        meta = ["Version", "PM", "Pop_Type", "Employment (1k)", "EPE"]
        econ = ["Profit", "Margin", "Input_Cost", "Output_Val", "Worker_Food_Cost", "output_price"]
        goods = sorted([c for c in combined.columns if c not in meta + econ])
        
        final_df = combined[meta + econ + goods].fillna(0)
        
        # Explicitly format all numeric columns to 3 decimal places
        float_cols = final_df.select_dtypes(include=[np.number]).columns.tolist()
        format_dict = {col: "{:.3f}" for col in float_cols}
        if 'Margin' in format_dict: 
            format_dict['Margin'] = "{:+.3%}"
        if 'EPE' in format_dict:
            format_dict['EPE'] = "{:+.3%}"
            
        styled = final_df.style.format(format_dict)
        
        # Add gradients for Margin, Profit and EPE
        if 'Margin' in final_df.columns:
            styled = styled.background_gradient(subset=['Margin'], cmap='RdYlGn', vmin=-0.5, vmax=0.5)
        if 'EPE' in final_df.columns:
            # For EPE, positive means we need more efficiency (bad current state), negative means we are already above break-even
            # So we invert the colormap: negative EPE is Green, positive is Red
            styled = styled.background_gradient(subset=['EPE'], cmap='RdYlGn_r', vmin=-0.5, vmax=0.5)
        if 'Profit' in final_df.columns:
            # For profit, we use a divergent map centered around 0
            # We'll use the max absolute profit to center the scale
            max_prof = final_df['Profit'].abs().max()
            styled = styled.background_gradient(subset=['Profit'], cmap='RdYlGn', vmin=-max_prof, vmax=max_prof)
        
        slot_dfs.append(styled)
    return slot_dfs

## Farming Village

In [37]:
farming_village_analysis = analyze_building("farming_village")
for df in farming_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_clay,In_debug_max_profit,In_horses,In_tools,Out_livestock,Out_millet,Out_wheat,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,farming_village_maintenance,peasants,1.000,-16.667%,0.125,+20.000%,0.625,0.750,0.050,1.500,1.000,0.600,0.000,0.025,0.500,0.000,0.000,0.625,0.000,0.750,0.125,0.200,0.050
1,Modded,pp_farming_village_maintenance,peasants,1.000,-16.667%,0.125,+20.000%,0.625,0.750,0.050,1.500,1.000,0.600,0.000,0.025,0.500,0.000,0.000,0.625,0.000,0.750,0.125,0.200,0.050
2,Modded,millet_farm_maintenance,peasants,1.000,+30.000%,-0.150,-23.077%,0.650,0.500,0.050,1.000,0.000,0.600,0.100,0.100,0.000,0.500,0.000,0.650,0.000,0.500,-0.150,-0.231,0.050
3,Modded,wheat_farm_maintenance,peasants,1.000,+62.500%,-0.250,-38.462%,0.650,0.400,0.050,1.000,0.000,0.600,0.100,0.100,0.000,0.000,0.400,0.650,0.000,0.400,-0.250,-0.385,0.050


## Fishing Village

In [38]:
fishing_village_analysis = analyze_building("fishing_village")
for df in fishing_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_debug_max_profit,In_ivory,In_salt,In_tools,Out_fish,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,fishing_village_maintenance,peasants,1.000,-14.000%,0.070,+16.279%,0.430,0.500,0.050,1.000,0.600,0.000,0.080,0.020,0.500,0.430,0.000,0.500,0.070,0.163,0.050
1,Vanilla,arctic_fishing_village_maintenance,peasants,1.000,-18.000%,0.090,+21.951%,0.410,0.500,0.050,1.000,0.600,0.010,0.080,0.000,0.500,0.410,0.000,0.500,0.090,0.220,0.050
2,Modded,fishing_village_maintenance,peasants,1.000,-14.000%,0.070,+16.279%,0.430,0.500,0.050,1.000,0.600,0.000,0.080,0.020,0.500,0.430,0.000,0.500,0.070,0.163,0.050
3,Modded,arctic_fishing_village_maintenance,peasants,1.000,-18.000%,0.090,+21.951%,0.410,0.500,0.050,1.000,0.600,0.010,0.080,0.000,0.500,0.410,0.000,0.500,0.090,0.220,0.050


## Forest Village

In [39]:
forest_village_analysis = analyze_building("forest_village")
for df in forest_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_debug_max_profit,In_sand,In_tar,In_wild_game,Out_food,Out_leather,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,forest_village_maintenance,peasants,1.000,-32.500%,0.260,+48.148%,0.540,0.800,0.050,3.000,0.600,0.100,0.020,0.400,1.000,0.250,0.540,1.000,0.800,0.260,0.481,0.050
1,Modded,forest_village_maintenance,peasants,1.000,-22.857%,0.160,+29.630%,0.540,0.700,0.050,3.000,0.600,0.100,0.020,0.400,-1.000,0.250,0.540,-1.000,0.700,0.160,0.296,0.050


## Fruit Orchard

In [40]:
fruit_orchard_analysis = analyze_building("fruit_orchard")
for df in fruit_orchard_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_lumber,Out_fruit,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,fruit_orchard_maintenance,laborers,1.000,+1.667%,-0.005,-1.639%,0.305,0.300,0.050,1.000,0.170,0.300,0.305,0.000,0.300,-0.005,-0.016,0.050
1,Modded,fruit_orchard_maintenance,peasants,1.000,+1.667%,-0.005,-1.639%,0.305,0.300,0.050,1.000,0.170,0.300,0.305,0.000,0.300,-0.005,-0.016,0.050


## Sheep Farm

In [41]:
sheep_farms_analysis = analyze_building("sheep_farms")
for df in sheep_farms_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_lumber,In_tools,Out_wool,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,sheep_farm_maintenance,laborers,1.000,-9.600%,0.120,+10.619%,1.130,1.250,0.050,2.500,0.400,0.160,0.500,1.130,0.000,1.250,0.120,0.106,0.050
1,Modded,sheep_farm_maintenance,peasants,1.000,-9.600%,0.120,+10.619%,1.130,1.250,0.050,2.500,0.400,0.160,0.500,1.130,0.000,1.250,0.120,0.106,0.050


## Cookery

In [42]:
cookery_analysis = analyze_building("cookery")
for df in cookery_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_fish,In_fruit,In_legumes,In_livestock,In_maize,In_millet,In_pepper,In_potato,In_rice,In_salt,In_wheat,In_wild_game,In_wool,Out_victuals,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Modded,cookery_rice_legumes,peasants,1.000,+9.179%,-0.317,-8.407%,3.767,3.450,0.017,3.000,0.000,0.000,1.500,0.000,0.000,0.000,0.100,0.000,1.750,0.000,0.000,0.000,0.000,1.150,3.767,0.000,3.450,-0.317,-0.084,0.017
1,Modded,cookery_rice_fish,peasants,1.000,+9.179%,-0.317,-8.407%,3.767,3.450,0.017,3.000,1.500,0.000,0.000,0.000,0.000,0.000,0.100,0.000,1.750,0.000,0.000,0.000,0.000,1.150,3.767,0.000,3.450,-0.317,-0.084,0.017
2,Modded,cookery_wheat_livestock,peasants,1.000,+13.527%,-0.467,-11.915%,3.917,3.450,0.017,3.000,0.000,0.000,0.000,1.400,0.000,0.000,0.000,0.000,0.000,0.100,1.400,0.000,0.000,1.150,3.917,0.000,3.450,-0.467,-0.119,0.017
3,Modded,cookery_millet_fish,peasants,1.000,+13.527%,-0.467,-11.915%,3.917,3.450,0.017,3.000,2.000,0.000,0.000,0.000,0.000,1.500,0.000,0.000,0.000,0.100,0.000,0.000,0.000,1.150,3.917,0.000,3.450,-0.467,-0.119,0.017
4,Modded,cookery_maize_livestock,peasants,1.000,+12.077%,-0.417,-10.776%,3.867,3.450,0.017,3.000,0.000,0.000,0.000,1.500,1.100,0.000,0.000,0.500,0.000,0.000,0.000,0.000,0.000,1.150,3.867,0.000,3.450,-0.417,-0.108,0.017
5,Modded,cookery_pemmican,peasants,1.000,+7.729%,-0.267,-7.175%,3.717,3.450,0.017,3.000,0.000,0.200,0.000,2.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.500,0.000,1.150,3.717,0.000,3.450,-0.267,-0.072,0.017
6,Modded,cookery_mutton_pease,peasants,1.000,+7.729%,-0.267,-7.175%,3.717,3.450,0.017,3.000,0.000,0.000,0.800,0.000,0.000,0.000,0.000,0.000,0.000,0.100,0.000,0.000,1.000,1.150,3.717,0.000,3.450,-0.267,-0.072,0.017
7,Modded,cookery_labskaus,peasants,1.000,+7.729%,-0.267,-7.175%,3.717,3.450,0.017,3.000,0.800,0.000,0.000,1.600,0.000,0.000,0.000,0.500,0.000,0.000,0.000,0.000,0.000,1.150,3.717,0.000,3.450,-0.267,-0.072,0.017
8,Modded,cookery_surstromming,peasants,1.000,+9.179%,-0.317,-8.407%,3.767,3.450,0.017,3.000,2.500,0.000,0.000,0.000,0.000,0.250,0.000,0.000,0.000,0.250,0.000,0.000,0.000,1.150,3.767,0.000,3.450,-0.317,-0.084,0.017


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_beer,In_beeswax,In_cloves,In_fruit,In_liquor,In_olives,In_pepper,In_pottery,In_salt,In_sugar,In_wine,Out_victuals,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Modded,cookery_no_enhancement,peasants,1.000,+0.000%,-0.017,-100.000%,0.017,0.000,0.017,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.017,0.000,0.000,-0.017,-1.000,0.017
1,Modded,cookery_preserved_rations,peasants,1.000,+67.778%,-1.017,-40.397%,2.517,1.500,0.017,3.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.900,0.400,0.000,0.000,0.500,2.517,0.000,1.500,-1.017,-0.404,0.017
2,Modded,cookery_sweet_preserves,peasants,1.000,+15.079%,-0.317,-13.103%,2.417,2.100,0.017,3.000,0.000,0.000,0.000,0.900,0.000,0.000,0.000,0.000,0.000,0.500,0.000,0.700,2.417,0.000,2.100,-0.317,-0.131,0.017
3,Modded,cookery_spice_blend,peasants,1.000,+67.460%,-1.417,-40.284%,3.517,2.100,0.017,3.000,0.000,0.000,0.350,0.000,0.000,0.000,0.350,0.000,0.000,0.000,0.000,0.700,3.517,0.000,2.100,-1.417,-0.403,0.017
4,Modded,cookery_spirited_rations,peasants,1.000,+87.037%,-1.567,-46.535%,3.367,1.800,0.017,3.000,0.800,0.000,0.000,0.000,0.700,0.000,0.000,0.000,0.000,0.000,0.000,0.600,3.367,0.000,1.800,-1.567,-0.465,0.017
5,Modded,cookery_mediterranean_preserves,peasants,1.000,+87.778%,-1.317,-46.746%,2.817,1.500,0.017,3.000,0.000,0.000,0.000,0.000,0.000,0.800,0.000,0.000,0.000,0.000,1.000,0.500,2.817,0.000,1.500,-1.317,-0.467,0.017
6,Modded,cookery_honey_confections,peasants,1.000,+45.370%,-0.817,-31.210%,2.617,1.800,0.017,3.000,0.000,1.000,0.000,0.600,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.600,2.617,0.000,1.800,-0.817,-0.312,0.017


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_steel,Out_victuals,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Modded,cookery_basic_packaging,peasants,1.000,+0.000%,-0.017,-100.000%,0.017,0.000,0.017,0.000,0.000,0.000,0.017,0.000,0.000,-0.017,-1.000,0.017
1,Modded,cookery_tin_cans,peasants,1.000,+109.722%,-1.317,-52.318%,2.517,1.200,0.017,3.000,0.500,0.400,2.517,0.000,1.200,-1.317,-0.523,0.017


## Tavern

In [43]:
tavern_analysis = analyze_building("tavern")
for df in tavern_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_victuals,Out_food,input_cost,is_modifier_output,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Modded,tavern_maintenance,peasants,1.000,+21.000%,-1.050,-17.355%,6.050,5.000,0.050,0.050,2.000,100.000,6.050,True,100.000,5.000,-1.050,-0.174,0.050


In [44]:
# Tavern calculator: set parameters and run to get the same-style DF
victuals = 2.0
food_produced = 120.0
employment = 1.0
v_good = goods_data.get_good("victuals") if goods_data else None
victuals_price = float(v_good.get("default_market_price", 3.0)) if v_good is not None else 3.0
production_efficiency = 0.0

# Tavern output is abstract food; value at FOOD_PRICE (defines), not market price
food_good_name = "food"
food_price = float(defines_data.get_define("NMarket", "FOOD_PRICE", 0.05)) if defines_data else 0.05
pop_info = pop_data.get_pop_type("peasants") if pop_data else None
pop_food_consumption = float(pop_info.get("pop_food_consumption", 0.6)) if pop_info is not None else 0.6

worker_food_cost = employment * pop_food_consumption * food_price
victuals_cost = victuals * victuals_price
input_cost = victuals_cost + worker_food_cost
output_value = food_produced * (1 + production_efficiency) * food_price
profit = output_value - input_cost
margin = (output_value / input_cost) - 1 if input_cost > 0 else 0.0
epe = (input_cost / (food_produced * food_price)) - 1 if (food_produced * food_price) > 0 else 0.0

out_col = f"Out_{food_good_name}"
tavern_calc_df = pd.DataFrame([{
    "Version": "Calculator",
    "PM": "tavern_maintenance",
    "Pop_Type": "peasants",
    "Employment (1k)": employment,
    "EPE": epe,
    "Profit": profit,
    "Margin": margin,
    "Input_Cost": input_cost,
    "Output_Val": output_value,
    "Worker_Food_Cost": worker_food_cost,
    "output_price": food_price,
    "In_victuals": victuals,
    out_col: food_produced * (1 + production_efficiency)
}])

meta = ["Version", "PM", "Pop_Type", "Employment (1k)", "EPE"]
econ = ["Profit", "Margin", "Input_Cost", "Output_Val", "Worker_Food_Cost", "output_price"]
goods = ["In_victuals", out_col]
display(tavern_calc_df[meta + econ + goods].fillna(0))


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_victuals,Out_food
0,Calculator,tavern_maintenance,peasants,1.00,0.01,-0.05,-0.01,6.05,6.00,0.05,0.05,2.00,120.00


In [45]:
# I think the tavern should be +- 0 at food baseline price? This also means that all global food multipliers will benefit tavern and subsistence
